# CI Payment Details 2 Monthly Burn EDA

This notebook analyzes `ci_payment_details_2.csv`, focusing on the `_months` model fields as requested. `_pg` fields are retained in the inventory but are not used for primary interpretation.

## Brief Analysis and Assumptions

The dataset has 16,435 posting rows for 875 contract items.

Every field is populated in this extract; missingness is not the main issue.

The monthly delta formulas are internally consistent: 16,435 absolute-delta rows and 16,435 percent-delta rows match the stated formulas.

The monthly linear model is a poor row-level predictor for many postings: median `BURN_DELTA_MONTHS_PERCENT` is -77.60%, and only 1,574 rows are within +/-25% of linear.

The distribution is asymmetric and bursty: 986 rows exceed +100%, while 126 rows are below -100%.

Negative actual burn exists in 126 rows, so reversals/corrections materially affect the lower tail.

Only 875 of 875 items have row count equal to `NUM_PAYGROUPS`; only 875 have distinct posting-date count equal to `NUM_PAYGROUPS`. This is the main grain caveat.

The item-level monthly rate reconciles to item total burn for 0 of 875 items using `(DAYS_BETWEEN + 30) / 30`, confirming the broad monthly-rate construction.

Clarifying questions carried as assumptions for this pass:

- Should repeated `ITEMID` + `WPPOSTINGDATE` rows be collapsed to one pay group, or retained as separate posting/detail rows? This notebook retains them and separately reports duplicate date groups.
- Should negative `THIS_BURN` values be treated as valid correction/reversal postings? This notebook treats them as valid and profiles their impact.
- Is `THIS_BURN` an amount for the posting row rather than a pre-normalized monthly rate? This notebook interprets it as the actual row burn compared to the monthly-linear model.

In [ ]:
import csv
from collections import Counter, defaultdict
from decimal import Decimal
from datetime import datetime
from pathlib import Path

CSV_PATH = Path('ci_payment_details_2.csv')
with CSV_PATH.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    rows = list(reader)
    fields = reader.fieldnames

print(f'Rows: {len(rows):,}')
print(f'Fields: {len(fields):,}')
print(fields)

## Field Inventory

| Field | Role | Missing | Missing % | Unique | Examples | Interpretation |
| --- | --- | --- | --- | --- | --- | --- |
| ITEMID | identifier/count | 0 | 0.00% | 875 | 18384; 23575; 17684 | Contract item identifier. User described each ITEMID as a particular contract item with multiple payment postings. |
| WPPOSTINGDATE | date | 0 | 0.00% | 3,628 | 2017-07-15 06:00:00.000; 2018-06-15 06:00:00.000; 2017-07-28 06:00:00.000 | Posting date for this actual burn observation. |
| FIRSTPOSTINGDATE | date | 0 | 0.00% | 659 | 2017-06-01 06:00:00.000; 2017-10-19 06:00:00.000; 2017-03-24 06:00:00.000 | First posting date observed for the ITEMID. |
| LASTPOSTINGDATE | date | 0 | 0.00% | 650 | 2017-07-17 06:00:00.000; 2018-08-28 06:00:00.000; 2017-10-26 06:00:00.000 | Last posting date observed for the ITEMID. |
| NUM_PAYGROUPS | identifier/count | 0 | 0.00% | 69 | 31; 11; 8 | Total number of postings/pay groups for the ITEMID according to the query. |
| DAYS_BETWEEN | identifier/count | 0 | 0.00% | 439 | 76; 343; 246 | Days between first and last posting for the ITEMID. |
| CI_BURN_RATE_PG | pay-group model | 0 | 0.00% | 872 | 89317.176000000000; 48062.418545454545; 194677.697500000000 | Linear item burn rate using fixed pay-group units. Ignored for primary interpretation in this notebook. |
| CI_BURN_RATE_MONTHS | monthly model | 0 | 0.00% | 875 | 1092960.323810568922; 46240.812193609685; 189929.460975609756 | Linear item burn rate using elapsed months: roughly item total burn / ((DAYS_BETWEEN + 30) / 30). |
| THIS_BURN | actual burn | 0 | 0.00% | 14,995 | 123108.55440000; 7153.66800000; 155742.15800000 | Actual burn amount/rate for this row. |
| BURN_DELTA_MONTHS_PERCENT | monthly model | 0 | 0.00% | 15,161 | -88.736228414000; -84.529536440600; -18.000000000000 | Monthly-model normalized deviation: (THIS_BURN - CI_BURN_RATE_MONTHS) / CI_BURN_RATE_MONTHS * 100. |
| BURN_DELTA_PG_PERCENT | pay-group model | 0 | 0.00% | 14,724 | 37.833012544000; -85.115880106500; -20.000000000000 | Pay-group-model normalized deviation. Ignored for primary interpretation in this notebook. |
| BURN_DELTA_PG | pay-group model | 0 | 0.00% | 15,267 | 33791.378400000000; -40908.750545454545; -38935.539500000000 | Absolute delta from pay-group linear model. Ignored for primary interpretation in this notebook. |
| BURN_DELTA_MONTHS | monthly model | 0 | 0.00% | 15,290 | -969851.769410568922; -39087.144193609685; -34187.302975609756 | Absolute delta from monthly linear model: THIS_BURN - CI_BURN_RATE_MONTHS. |

## Numeric Ranges

| Field | Parsed | Missing | Parse failures | Min | P05 | P25 | Median | P75 | P95 | Max | Sum | Mean |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| ITEMID | 16,435 | 0 | 0 | 4,164.0000 | 19,677.0000 | 38,932.0000 | 57,754.0000 | 89,748.0000 | 106,139.0000 | 116,870.0000 | 1,014,978,482.00 | 61,757.1331 |
| NUM_PAYGROUPS | 16,435 | 0 | 0 | 7.0000 | 8.0000 | 14.0000 | 24.0000 | 39.0000 | 118.0000 | 213.0000 | 575,881.00 | 35.0399 |
| DAYS_BETWEEN | 16,435 | 0 | 0 | 36.0000 | 51.0000 | 105.0000 | 320.0000 | 575.0000 | 1,054.0000 | 1,402.0000 | 6,404,842.00 | 389.7075 |
| CI_BURN_RATE_MONTHS | 16,435 | 0 | 0 | 11,463.6981 | 29,561.0880 | 74,792.6603 | 207,983.3718 | 580,401.2714 | 1,583,150.0447 | 10,226,006.4134 | 8,173,711,668.25 | 497,335.6659 |
| THIS_BURN | 16,435 | 0 | 0 | -929,900.0000 | 1,711.7215 | 19,377.0035 | 65,215.7471 | 141,242.4468 | 439,212.7998 | 30,600,018.7090 | 3,917,994,164.93 | 238,393.3170 |
| BURN_DELTA_MONTHS_PERCENT | 16,435 | 0 | 0 | -756.1830 | -98.3254 | -89.3992 | -77.5994 | -38.1378 | 122.2578 | 1,834.6319 | -714,873.33 | -43.4970 |
| BURN_DELTA_MONTHS | 16,435 | 0 | 0 | -9,982,342.4134 | -1,218,098.5782 | -399,690.3805 | -122,610.0000 | -24,401.0908 | 117,255.9233 | 23,159,344.9328 | -4,255,717,503.32 | -258,942.3488 |
| CI_BURN_RATE_PG | 16,435 | 0 | 0 | 3,788.6091 | 8,611.8153 | 37,527.1402 | 81,377.6216 | 135,943.0372 | 519,047.6190 | 11,094,405.4519 | 3,917,994,164.93 | 238,393.3170 |
| BURN_DELTA_PG_PERCENT | 16,435 | 0 | 0 | -1,371.3300 | -94.4267 | -52.0000 | -7.5472 | 29.5224 | 128.7088 | 2,100.2335 | 0.00 | 0.0000 |
| BURN_DELTA_PG | 16,435 | 0 | 0 | -10,850,741.4519 | -130,666.7500 | -29,714.2955 | -3,020.9449 | 22,352.6362 | 121,181.2868 | 23,040,725.5497 | 0.00 | -0.0000 |

## Date Ranges

| Field | Parsed | Missing | Parse failures | Min | Median | Max |
| --- | --- | --- | --- | --- | --- | --- |
| WPPOSTINGDATE | 16,435 | 0 | 0 | 2016-08-30 06:00:00 | 2021-04-19 06:00:00 | 2026-04-30 06:00:00 |
| FIRSTPOSTINGDATE | 16,435 | 0 | 0 | 2016-08-30 06:00:00 | 2020-07-24 06:00:00 | 2025-12-07 07:00:00 |
| LASTPOSTINGDATE | 16,435 | 0 | 0 | 2016-10-07 06:00:00 | 2021-11-19 07:00:00 | 2026-04-30 06:00:00 |

## Internal Consistency Checks

| Check | Pass | Fail |
| --- | --- | --- |
| BURN_DELTA_MONTHS = THIS_BURN - CI_BURN_RATE_MONTHS | 16,435 | 0 |
| BURN_DELTA_MONTHS_PERCENT formula | 16,435 | 0 |
| WPPOSTINGDATE within FIRST/LAST span | 16,435 | 0 |
| FIRSTPOSTINGDATE <= LASTPOSTINGDATE | 16,435 | 0 |
| DAYS_BETWEEN agrees with FIRST/LAST dates | 0 | 16,435 |

Exact duplicate rows: 0. Duplicate `ITEMID` + `WPPOSTINGDATE` groups: 0.

## Row Grain and Paygroup Count

`NUM_PAYGROUPS` behaves like an item-level count from the query rather than a guaranteed count of rows in this export. This matters because repeated dates or row-level detail splits can make a posting group appear more than once.

| NUM_PAYGROUPS | Item count | Share of items |
| --- | --- | --- |
| 7 | 103 | 11.77% |
| 8 | 82 | 9.37% |
| 9 | 69 | 7.89% |
| 10 | 51 | 5.83% |
| 11 | 60 | 6.86% |
| 12 | 39 | 4.46% |
| 13 | 34 | 3.89% |
| 14 | 39 | 4.46% |
| 15 | 31 | 3.54% |
| 16 | 31 | 3.54% |
| 17 | 22 | 2.51% |
| 18 | 26 | 2.97% |
| 19 | 25 | 2.86% |
| 20 | 18 | 2.06% |
| 21 | 15 | 1.71% |
| 22 | 11 | 1.26% |
| 23 | 17 | 1.94% |
| 24 | 8 | 0.91% |
| 25 | 15 | 1.71% |
| 26 | 19 | 2.17% |
| 27 | 15 | 1.71% |
| 28 | 10 | 1.14% |
| 29 | 13 | 1.49% |
| 30 | 11 | 1.26% |
| 31 | 9 | 1.03% |
| 32 | 5 | 0.57% |
| 33 | 5 | 0.57% |
| 34 | 6 | 0.69% |
| 35 | 8 | 0.91% |
| 36 | 4 | 0.46% |
| 37 | 3 | 0.34% |
| 38 | 4 | 0.46% |
| 39 | 6 | 0.69% |
| 40 | 3 | 0.34% |
| 41 | 6 | 0.69% |
| 42 | 5 | 0.57% |
| 43 | 1 | 0.11% |
| 44 | 1 | 0.11% |
| 45 | 2 | 0.23% |
| 46 | 1 | 0.11% |
| 47 | 1 | 0.11% |
| 48 | 2 | 0.23% |
| 50 | 1 | 0.11% |
| 51 | 2 | 0.23% |
| 53 | 1 | 0.11% |
| 54 | 1 | 0.11% |
| 56 | 3 | 0.34% |
| 57 | 1 | 0.11% |
| 59 | 4 | 0.46% |
| 61 | 3 | 0.34% |
| 65 | 2 | 0.23% |
| 66 | 2 | 0.23% |
| 67 | 1 | 0.11% |
| 68 | 2 | 0.23% |
| 70 | 1 | 0.11% |
| 74 | 1 | 0.11% |
| 75 | 2 | 0.23% |
| 76 | 1 | 0.11% |
| 77 | 1 | 0.11% |
| 87 | 1 | 0.11% |
| 93 | 1 | 0.11% |
| 95 | 1 | 0.11% |
| 101 | 1 | 0.11% |
| 118 | 1 | 0.11% |
| 126 | 1 | 0.11% |
| 141 | 1 | 0.11% |
| 147 | 1 | 0.11% |
| 165 | 1 | 0.11% |
| 213 | 1 | 0.11% |

Items where row count equals `NUM_PAYGROUPS`: 875 of 875.

Items where distinct posting dates equal `NUM_PAYGROUPS`: 875 of 875.

## Monthly Delta Distribution

| Delta percent bucket | Rows | Share |
| --- | --- | --- |
| < -100% | 126 | 0.77% |
| -100% to -75% | 8,632 | 52.52% |
| -75% to -50% | 2,782 | 16.93% |
| -50% to -25% | 1,335 | 8.12% |
| -25% to +25% | 1,574 | 9.58% |
| +25% to +100% | 1,000 | 6.08% |
| > +100% | 986 | 6.00% |

The large mass below zero means most posting rows are below the even monthly allocation. The positive tail indicates concentrated/bursty postings that exceed the linear monthly model by multiples.

## Delta by Number of Paygroups

| NUM_PAYGROUPS bucket | Rows | P25 delta % | Median delta % | P75 delta % | P95 delta % | Total this burn | Negative burn share |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 07-08 | 1,377 | -78.16 | -34.01 | 52.00 | 311.50 | 205,671,476.46 | 0.87% |
| 09-12 | 2,259 | -84.33 | -67.25 | 2.00 | 216.34 | 297,643,363.75 | 0.97% |
| 13-18 | 2,791 | -87.73 | -74.73 | -25.38 | 150.47 | 359,003,743.63 | 1.22% |
| 19-30 | 4,236 | -89.51 | -79.08 | -43.22 | 75.59 | 1,066,539,070.19 | 0.71% |
| 31+ | 5,772 | -92.41 | -83.36 | -62.33 | 28.10 | 1,989,136,510.91 | 0.49% |

## Delta by Item Duration

| DAYS_BETWEEN bucket | Rows | P25 delta % | Median delta % | P75 delta % | P95 delta % | Total this burn | Negative burn share |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1095+ | 732 | -79.69 | -34.52 | 55.29 | 219.17 | 1,594,287,075.86 | 1.50% |
| 180-364 | 3,186 | -83.71 | -63.93 | -25.49 | 97.94 | 334,363,374.20 | 1.04% |
| 365-729 | 4,312 | -85.27 | -55.39 | 2.61 | 208.40 | 455,341,337.91 | 0.56% |
| 730-1094 | 2,048 | -81.88 | -53.91 | 14.36 | 217.25 | 893,608,856.89 | 1.17% |
| <180 | 6,157 | -91.94 | -86.86 | -79.92 | -54.79 | 640,393,520.08 | 0.55% |

## Top Positive Monthly Delta Rows

These are the posting rows most above the monthly-linear model.

| ITEMID | WPPOSTINGDATE | NUM_PAYGROUPS | DAYS_BETWEEN | CI_BURN_RATE_MONTHS | THIS_BURN | BURN_DELTA_MONTHS_PERCENT | BURN_DELTA_MONTHS |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 19672 | 2017-05-06 06:00:00.000 | 15 | 942 | 107615.573248407643 | 2081965.25000000 | 1834.631937697600 | 1974349.676751592357 |
| 19670 | 2017-05-06 06:00:00.000 | 12 | 942 | 100318.471337579618 | 1922666.51000000 | 1816.562806793700 | 1822348.038662420382 |
| 61751 | 2020-05-15 06:00:00.000 | 11 | 1234 | 64916.348500132484 | 1229991.00000000 | 1794.732264550400 | 1165074.651499867516 |
| 44983 | 2019-11-16 07:00:00.000 | 11 | 905 | 39701.768378985985 | 692580.00000000 | 1644.456300759100 | 652878.231621014015 |
| 57300 | 2020-07-27 06:00:00.000 | 9 | 1293 | 196851.902552204176 | 3381031.00000000 | 1617.550583034600 | 3184179.097447795824 |
| 68867 | 2021-08-21 06:00:00.000 | 8 | 758 | 26145.418388582871 | 424836.73469800 | 1524.899354770000 | 398691.316309417129 |
| 75397 | 2022-03-19 06:00:00.000 | 13 | 561 | 39326.763636363636 | 630099.69926400 | 1502.216000000000 | 590772.935627636364 |
| 80920 | 2022-04-16 06:00:00.000 | 7 | 520 | 123351.811795227150 | 1974747.34050800 | 1500.906635880000 | 1851395.528712772850 |
| 71522 | 2021-05-21 06:00:00.000 | 14 | 1234 | 146645.057914465623 | 2179270.00000000 | 1386.084857541600 | 2032624.942085534377 |
| 36218 | 2019-01-18 07:00:00.000 | 68 | 1349 | 11463.698073619733 | 166792.03300000 | 1354.958355749300 | 155328.334926380267 |
| 73938 | 2021-08-21 06:00:00.000 | 13 | 723 | 47238.506224066390 | 681816.50720000 | 1343.349000000000 | 634578.000975933610 |
| 57300 | 2020-09-25 06:00:00.000 | 9 | 1293 | 196851.902552204176 | 2738048.20000000 | 1290.917824263300 | 2541196.297447795824 |
| 63638 | 2021-05-15 06:00:00.000 | 38 | 1192 | 167406.551823880468 | 2236218.42000000 | 1235.801015931900 | 2068811.868176119532 |
| 87872 | 2023-08-28 06:00:00.000 | 9 | 612 | 69283.854166666667 | 769635.90000000 | 1010.844523961700 | 700352.045833333333 |
| 36921 | 2019-08-16 06:00:00.000 | 14 | 673 | 58218.213673376132 | 630584.03110000 | 983.138748704600 | 572365.817426623868 |
| 19669 | 2017-05-06 06:00:00.000 | 14 | 942 | 70700.636942675159 | 762775.17000000 | 978.880195405400 | 692074.533057324841 |
| 64963 | 2020-11-21 07:00:00.000 | 13 | 632 | 36966.051155600456 | 389375.74500000 | 953.333350000000 | 352409.693844399544 |
| 19671 | 2017-05-06 06:00:00.000 | 15 | 942 | 78025.477707006369 | 796647.17000000 | 921.009026040800 | 718621.692292993631 |
| 51375 | 2019-10-26 06:00:00.000 | 15 | 1201 | 18109.908560448864 | 181250.00000000 | 900.833325000000 | 163140.091439551136 |
| 87855 | 2023-08-25 06:00:00.000 | 8 | 802 | 60893.880123365089 | 599103.75000000 | 883.848867548400 | 538209.869876634911 |

## Top Negative Monthly Delta Rows

These rows are most below the monthly-linear model. Values below -100% are possible because actual burn can be negative.

| ITEMID | WPPOSTINGDATE | NUM_PAYGROUPS | DAYS_BETWEEN | CI_BURN_RATE_MONTHS | THIS_BURN | BURN_DELTA_MONTHS_PERCENT | BURN_DELTA_MONTHS |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 75397 | 2022-01-15 07:00:00.000 | 13 | 561 | 39326.763636363636 | -258055.53743200 | -756.183000000000 | -297382.301068363636 |
| 57300 | 2023-09-12 06:00:00.000 | 9 | 1293 | 196851.902552204176 | -851893.50000000 | -532.758580920500 | -1048745.402552204176 |
| 21164 | 2018-03-14 06:00:00.000 | 25 | 242 | 163732.431250725981 | -671656.35000000 | -510.215828879700 | -835388.781250725981 |
| 77138 | 2022-10-29 06:00:00.000 | 9 | 119 | 267539.221215191495 | -929900.00000000 | -447.575206273000 | -1197439.221215191495 |
| 58603 | 2020-11-09 07:00:00.000 | 12 | 851 | 57101.153265556366 | -160357.17060000 | -380.830003300000 | -217458.323865556366 |
| 36188 | 2022-03-07 07:00:00.000 | 23 | 1289 | 27439.422809779497 | -75728.38446000 | -375.983882696700 | -103167.807269779497 |
| 17383 | 2018-07-21 06:00:00.000 | 30 | 594 | 69641.467171717172 | -188903.64600000 | -371.251674715900 | -258545.113171717172 |
| 63537 | 2021-01-20 07:00:00.000 | 8 | 321 | 51401.869158878505 | -137500.00000000 | -367.500000000000 | -188901.869158878505 |
| 41320 | 2020-03-12 06:00:00.000 | 10 | 432 | 77356.082222222222 | -202736.01600000 | -362.081545724600 | -280092.098222222222 |
| 36218 | 2022-03-07 07:00:00.000 | 68 | 1349 | 11463.698073619733 | -29444.69063500 | -356.851588779700 | -40908.388708619733 |
| 19679 | 2018-05-19 06:00:00.000 | 27 | 898 | 121830.157704122023 | -304763.40000000 | -350.154317899000 | -426593.557704122023 |
| 17423 | 2018-12-27 07:00:00.000 | 24 | 667 | 26524.546400667862 | -61564.23980000 | -332.102893938500 | -88088.786200667862 |
| 44999 | 2020-09-29 06:00:00.000 | 66 | 980 | 24002.399755077554 | -48333.60000000 | -301.369865068500 | -72335.999755077554 |
| 99400 | 2025-01-16 07:00:00.000 | 18 | 184 | 284351.164774519825 | -524573.57250000 | -284.480894571300 | -808924.737274519825 |
| 92749 | 2025-03-14 06:00:00.000 | 45 | 530 | 420033.527546537216 | -693377.24000000 | -265.076641393400 | -1113410.767546537216 |
| 32544 | 2019-02-15 07:00:00.000 | 21 | 1031 | 28191.270337621044 | -44200.00000000 | -256.786123756200 | -72391.270337621044 |
| 89748 | 2025-09-02 06:00:00.000 | 213 | 746 | 34457.886937561837 | -50063.70000000 | -245.289524255300 | -84521.586937561837 |
| 45014 | 2020-04-17 06:00:00.000 | 22 | 809 | 22792.042208998242 | -27670.50000000 | -221.404215323300 | -50462.542208998242 |
| 47606 | 2020-01-07 07:00:00.000 | 59 | 356 | 42817.170145585108 | -47897.40000000 | -211.864936045800 | -90714.570145585108 |
| 61751 | 2021-05-22 06:00:00.000 | 11 | 1234 | 64916.348500132484 | -69276.22000000 | -206.716137923000 | -134192.568500132484 |

## Largest Items by Absolute Total Burn

| ITEMID | Rows | Unique posting dates | NUM_PAYGROUPS | DAYS_BETWEEN | Modeled months | CI burn rate months | Sum this burn | Min delta % | Median delta % | Max delta % | Negative rows |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 68698 | 42 | 42 | 42 | 1367 | 46.57 | 10,226,006.41 | 465,965,028.98 | -97.62 | 21.06 | 185.61 | 0 |
| 54059 | 46 | 46 | 46 | 1402 | 47.73 | 7,440,673.78 | 347,727,485.33 | -98.13 | -3.96 | 311.25 | 0 |
| 93523 | 29 | 29 | 29 | 948 | 32.60 | 6,831,868.47 | 215,887,043.60 | -97.63 | 6.73 | 136.40 | 0 |
| 42512 | 35 | 35 | 35 | 1219 | 41.63 | 5,201,006.13 | 211,334,214.00 | -100.00 | -3.53 | 229.41 | 0 |
| 29914 | 38 | 38 | 38 | 1256 | 42.87 | 4,369,703.52 | 182,944,921.96 | -101.69 | 22.01 | 118.15 | 1 |
| 96813 | 26 | 26 | 26 | 791 | 27.37 | 6,299,591.15 | 166,099,222.07 | -99.22 | 4.61 | 101.48 | 0 |
| 57754 | 32 | 32 | 32 | 1225 | 41.83 | 2,659,138.01 | 108,581,468.00 | -100.00 | -2.27 | 212.30 | 0 |
| 94749 | 26 | 26 | 26 | 809 | 27.97 | 3,919,235.93 | 105,688,730.22 | -91.59 | -10.79 | 142.96 | 0 |
| 32539 | 37 | 37 | 37 | 1129 | 38.63 | 2,560,043.51 | 96,342,969.86 | -98.38 | 10.27 | 135.24 | 0 |
| 44374 | 31 | 31 | 31 | 947 | 32.57 | 3,040,983.39 | 95,993,710.00 | -99.68 | -7.83 | 137.38 | 0 |
| 71520 | 34 | 34 | 34 | 1234 | 42.13 | 2,223,418.27 | 91,456,604.00 | -100.00 | 16.42 | 194.17 | 0 |
| 29913 | 38 | 38 | 38 | 1249 | 42.63 | 943,787.04 | 39,293,000.00 | -94.68 | -63.94 | 572.50 | 0 |
| 96809 | 26 | 26 | 26 | 790 | 27.33 | 1,320,664.57 | 34,777,500.00 | 1.28 | 1.28 | 1.28 | 0 |
| 96810 | 15 | 15 | 15 | 456 | 16.20 | 2,203,947.37 | 33,500,000.00 | -99.56 | 1.47 | 87.27 | 0 |
| 19674 | 26 | 26 | 26 | 898 | 30.93 | 916,290.70 | 27,427,634.54 | -103.46 | 19.37 | 217.98 | 1 |
| 19677 | 24 | 24 | 24 | 843 | 29.10 | 966,075.64 | 27,146,725.59 | -99.24 | 24.33 | 201.98 | 0 |
| 19675 | 26 | 26 | 26 | 898 | 30.93 | 892,689.37 | 26,721,168.31 | -99.46 | -69.34 | 533.43 | 0 |
| 19676 | 27 | 27 | 27 | 898 | 30.93 | 762,356.61 | 22,819,874.37 | -99.71 | -16.76 | 213.40 | 0 |
| 63936 | 27 | 27 | 27 | 1030 | 35.33 | 627,981.19 | 21,560,687.45 | -100.00 | 5.22 | 665.72 | 0 |
| 29911 | 35 | 35 | 35 | 1059 | 36.30 | 566,572.24 | 20,000,000.00 | 0.85 | 0.85 | 0.87 | 0 |
| 71521 | 32 | 32 | 32 | 968 | 33.27 | 581,962.80 | 18,778,000.00 | -21.62 | -21.62 | 239.65 | 0 |
| 106167 | 17 | 17 | 17 | 546 | 19.20 | 808,177.26 | 14,708,826.21 | -95.92 | 23.10 | 223.98 | 0 |
| 43164 | 77 | 77 | 77 | 397 | 14.23 | 1,071,380.67 | 14,177,937.16 | -98.01 | -82.45 | -68.62 | 0 |
| 91118 | 11 | 11 | 11 | 618 | 21.60 | 635,119.71 | 13,083,466.00 | -94.85 | 49.35 | 406.35 | 0 |
| 96814 | 19 | 19 | 19 | 638 | 22.27 | 603,291.53 | 12,830,000.00 | -99.78 | -33.51 | 368.63 | 0 |

## Most Volatile Items by Mean Absolute Delta Percent

| ITEMID | Rows | Unique posting dates | NUM_PAYGROUPS | DAYS_BETWEEN | Sum this burn | Mean abs delta % | Min delta % | Median delta % | Max delta % | Negative rows |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 57300 | 9 | 9 | 9 | 1293 | 8,484,317.00 | 497.97 | -532.76 | 212.60 | 1,617.55 | 1 |
| 61751 | 11 | 11 | 11 | 1234 | 2,670,225.78 | 347.89 | -206.72 | 217.41 | 1,794.73 | 2 |
| 57455 | 8 | 8 | 8 | 1053 | 612,500.00 | 346.20 | -29.80 | 251.00 | 847.70 | 0 |
| 89262 | 7 | 7 | 7 | 882 | 1,400,000.00 | 320.00 | 47.00 | 194.00 | 635.00 | 0 |
| 69763 | 7 | 7 | 7 | 856 | 990,000.00 | 307.62 | 42.67 | 185.33 | 613.33 | 0 |
| 69765 | 7 | 7 | 7 | 856 | 505,000.00 | 307.62 | 42.67 | 185.33 | 613.33 | 0 |
| 56910 | 8 | 8 | 8 | 841 | 671,088.00 | 289.39 | -100.00 | 292.60 | 623.44 | 0 |
| 80920 | 7 | 7 | 7 | 520 | 2,138,098.03 | 281.21 | -96.19 | -77.47 | 1,500.91 | 0 |
| 87855 | 8 | 8 | 8 | 802 | 1,627,896.38 | 274.29 | -119.74 | 88.39 | 883.85 | 1 |
| 68867 | 8 | 8 | 8 | 758 | 660,607.58 | 270.37 | -81.81 | 13.32 | 1,524.90 | 0 |
| 71522 | 14 | 14 | 14 | 1234 | 6,032,000.00 | 250.14 | -99.93 | 12.21 | 1,386.08 | 0 |
| 19670 | 12 | 12 | 12 | 942 | 3,150,000.00 | 239.47 | -95.02 | -21.75 | 1,816.56 | 0 |
| 91200 | 8 | 8 | 8 | 812 | 700,000.00 | 238.33 | 35.33 | 170.67 | 576.67 | 0 |
| 21019 | 8 | 8 | 8 | 718 | 512,753.41 | 237.20 | -100.00 | 175.23 | 546.20 | 0 |
| 44983 | 11 | 11 | 11 | 905 | 1,197,670.03 | 228.17 | -75.72 | 27.42 | 1,644.46 | 0 |
| 57298 | 7 | 7 | 7 | 643 | 7,349,420.21 | 228.13 | -76.80 | 122.55 | 617.84 | 0 |
| 19672 | 15 | 15 | 15 | 942 | 3,379,129.00 | 220.45 | -137.90 | -57.84 | 1,834.63 | 2 |
| 75397 | 13 | 13 | 13 | 561 | 735,410.48 | 219.44 | -756.18 | -21.65 | 1,502.22 | 1 |
| 51368 | 9 | 9 | 9 | 699 | 1,323,927.50 | 214.13 | -77.05 | 285.21 | 416.50 | 0 |
| 58603 | 12 | 12 | 12 | 851 | 1,619,769.40 | 212.27 | -380.83 | 126.93 | 609.17 | 1 |
| 32534 | 13 | 13 | 13 | 1149 | 2,500,000.00 | 210.00 | -100.00 | 219.17 | 219.17 | 0 |
| 56007 | 7 | 7 | 7 | 534 | 882,714.31 | 209.81 | -193.68 | 98.98 | 460.83 | 1 |
| 90940 | 7 | 7 | 7 | 526 | 8,321,776.19 | 205.38 | -88.25 | 114.61 | 426.00 | 0 |
| 89271 | 7 | 7 | 7 | 639 | 532,380.00 | 204.29 | 30.97 | 208.35 | 397.44 | 0 |
| 51305 | 10 | 10 | 10 | 686 | 1,996,542.00 | 203.93 | -100.00 | -17.92 | 661.63 | 0 |

## Negative and Zero Burn Rows

| Metric | Value |
| --- | --- |
| Negative THIS_BURN rows | 126 |
| Zero THIS_BURN rows | 109 |
| Items with at least one negative row | 105 |

### Items With Most Negative Rows

| ITEMID | Negative rows | Negative row share | Item total this burn |
| --- | --- | --- | --- |
| 101515 | 4 | 22.22% | 1,149,619.18 |
| 92684 | 3 | 5.36% | 656,267.70 |
| 44978 | 3 | 2.13% | 10,422,319.14 |
| 87872 | 2 | 22.22% | 1,413,390.62 |
| 44916 | 2 | 5.71% | 1,581,307.20 |
| 45014 | 2 | 9.09% | 614,625.41 |
| 106139 | 2 | 9.52% | 2,267,369.90 |
| 18462 | 2 | 28.57% | 1,775,051.50 |
| 45139 | 2 | 11.11% | 576,894.50 |
| 19679 | 2 | 7.41% | 3,646,782.68 |
| 66704 | 2 | 15.38% | 3,996,738.02 |
| 36913 | 2 | 3.57% | 1,017,262.78 |
| 31170 | 2 | 15.38% | 4,122,840.90 |
| 61751 | 2 | 18.18% | 2,670,225.78 |
| 19672 | 2 | 13.33% | 3,379,129.00 |
| 26988 | 2 | 13.33% | 527,782.98 |
| 65938 | 2 | 18.18% | 537,154.80 |
| 20941 | 1 | 1.64% | 732,528.52 |
| 28398 | 1 | 3.33% | 2,489,953.33 |
| 54085 | 1 | 8.33% | 567,320.48 |
| 57300 | 1 | 11.11% | 8,484,317.00 |
| 35610 | 1 | 3.12% | 2,927,657.12 |
| 36211 | 1 | 3.03% | 541,522.47 |
| 31754 | 1 | 11.11% | 688,547.84 |
| 43464 | 1 | 7.14% | 1,026,276.52 |

## Posting-Date Spacing

| Metric | Count | Min | Median | Max | Mean |
| --- | --- | --- | --- | --- | --- |
| Inter-posting gap days | 15,560 | 0.00 | 2.00 | 1,025.04 | 16.22 |

### Duplicate ITEMID + WPPOSTINGDATE Groups

| ITEMID | WPPOSTINGDATE | Rows on same date |
| --- | --- | --- |

## Modeling Implications

- The monthly-linear model is useful as a baseline, but it should not be expected to predict individual posting rows closely. The empirical pattern is under-burn in many periods and sharp over-burn in fewer concentrated periods.
- `BURN_DELTA_MONTHS_PERCENT` is a better cross-item comparability field than the absolute delta, but it still explodes for low baseline rates and for negative/correction rows. Winsorized or bucketed views will likely be more stable than raw percent deltas for model training.
- Treat `ITEMID` as the primary grouping key. Any train/test split or validation should avoid splitting rows from the same item across train and test if the goal is generalization to unseen items.
- Before modeling payment timing, decide whether repeated `ITEMID` + `WPPOSTINGDATE` rows are separate observations or should be collapsed to a date/paygroup level. That choice changes the interpretation of `NUM_PAYGROUPS` and row weights.
- Negative burns should be modeled or flagged explicitly; excluding them would remove genuine correction behavior and would also hide why the percent delta can fall below -100%.